# Exercise 2: Use Puma to navigate in Google Maps by spoofing the location
In this exercise, you will navigate a route in Google Maps, by supplying a start and end location. You will be able to adjust the speed to the conditions during the run.

First, configure your start and end point by entering landmarks or addresses.

**Important:** Both the start and end locations must be recognizable **landmarks** (e.g., a well‑known building or attraction) or **exact addresses**. OpenStreetMap resolves these names to geographic coordinates, so ambiguous or non‑existent names will cause the routing request to fail or to result in the wrong location.

This is important because Puma will do a lookup in OpenStreetMaps in the background to perform location spoofing. If the given locations cannot be found on OSM, no route can be spoofed. Any of the following should work landmarks/buildings
 * Amsterdam Schiphol P1
 * Netherlands Forensic Institute
 * De Leeuwenhorst Noordwijk
 * Eiffel Tower
 * Your own work place
 * Any exact address (street + number + city)

Once the start and destination locations are set, the script will request a route from the OpenStreetMap **OSRM** service, parse the returned geometry, and feed it to the Puma `RouteSimulator`. This component will spoof the location of the device in such a way the device moves along the route at the desired speed.

Feel free to experiment with different start/end combinations to see how the simulated navigation behaves. Or think of an ambiguous name and see if you get the location you expected.  Enter your location in the `search()` below to check if the location can be correctly found.

In [ ]:
import urllib3

from puma.apps.android.google_maps.google_maps import GoogleMapsActions
maps = GoogleMapsActions(device_udid="emulator-5554")

In [ ]:
maps.search_place(search_string="Eiffel tower") # Enter here

Then, enter your start and end locations and start the navigation with a certain speed. Don't worry, you will be able to adjust the speed during the navigation.

In [ ]:
start_location = "Eiffel tower" # Enter here
end_location = "De Leeuwenhorst Noordwijk" # Enter here
route_simulator = maps.route_simulator
maps.start_route(from_query=start_location, to_query=end_location, speed=30)

#### Adjusting the speed
Adjust the speed by running the code block below. You can adjust the speed as many times as you want, for example when you exit the highway to adhere to the speed limit, or you can go crazy and try out speeds that would not be possible in real life without losing your driver's license!

In [ ]:
route_simulator.update_speed(80) # Enter speed (int)

If you do not want to spoof the location at a constant speed, there's optional arguments you can use. Refer to the `README.md` and see how you can use them.

## ❗Stop location spoofing before going to the next exercise❗

Otherwise, the location spoofing will continue in the background. This can be an issue if you want to use location spoofing in exercise 4.

In [ ]:
route_simulator.stop_route()

## ℹ️ How the navigation works
Puma’s navigation feature works by first asking the OpenStreetMap routing service for a path between the two places you provide. The service returns a series of latitude‑longitude coordinates that describe the road network between the start and end points. Puma stores this list of points in a small simulator component that can feed them to the device as if they were real GPS readings.
Once the route data is ready, Puma drives the Google Maps user interface: it opens the app, enters the destination, and taps the “Start” button to launch turn‑by‑turn navigation. At this moment Google Maps begins showing the route and navigation instructions, but because the device itself is stationary, nothing moves on the map.
The simulator then takes over. It repeatedly injects the stored coordinates into the device’s location provider, spacing the updates so that the virtual vehicle travels at the speed you specify. The speed value is converted into a time interval between successive points, and optional variance can be added to make the motion look more realistic. While the navigation UI continues to run, you can change the speed at any time; the simulator instantly adjusts the interval, allowing the vehicle to accelerate, decelerate, or stop without restarting.

When the sudden speed change is very big, you can see that the speed does not change instantly, but seems to smooth out. This is because Google Maps corrects for temporary loss of connection, for example when you are in a tunnel.